# Finding the left nullspace of the stiochiometry matrix
Details TBD...

Reference Padmini's paper https://onlinelibrary.wiley.com/doi/epdf/10.1002/bit.21307

In [ ]:
import pymc as pm
import matplotlib.pyplot as plt
import numpy as np
import sympy as sym
from sympy.solvers.solveset import linear_eq_to_matrix
from ampk_MA_double_mech import *
from sunode import dtypesubset, basic

In [2]:
# DEVELOPMENT VERSION!
# FINAL VERSION IN get_left_nullspace_vecs_invariants.py 

# Function that takes the right hand side of the model and returns the 
# stoichiometry matrix, a list of vectors in the left nullspace, and the 
# conservation laws in the model
def get_left_nullspace_vecs_invariants(rhs, sym_fluxes):
    """Function to find vectors in the left nullspace of a model stoichiometry matrix.

    Inputs: 
    - rhs: Dict[name, expr] of sympy expressions of the flux variables. E.g.,
        {'S1':J1-J2}
    - sym_fluxes: List of sympy expressions of the flux variables.

    Returns:
    - stoichiometry: sympy matrix with shape (num_states, num_fluxes) such that
        dXdt = stoichiometry * fluxes (X is vector of states and fluxes is vec of fluxes)
    - left_nulls: List of vectors in the left nullspace (nullspace of A^T) of
        the stoichiometry matrix
    - invariants: sympy matrices of inner product between left nullspace vectors
        and the state vector
    """
    # initialize states as sympy vars and store in a list
    state_names = rhs.keys()
    sym_states = []
    exprs = []
    for state in state_names:
        tmp = state.replace('_', '')
        exprs.append(rhs[state])
        exec('{i} = sym.symbols("{i}")'.format(i=tmp))
        exec('sym_states.append({i})'.format(i=tmp))

    # compute stoichiometry matrix by treating the RHS as a system of linear eqns    
    stoichiometry, _ = linear_eq_to_matrix(exprs, sym_fluxes)
    states = sym.Matrix(sym_states)

    # find left nullspace of stoichiometry matrix
    a = sym.Matrix(sym.Transpose(stoichiometry))
    left_nulls = a.nullspace()

    # compute list of invariants
    invariants = [ns_vec.transpose()*states for ns_vec in left_nulls]

    return stoichiometry, left_nulls, invariants 

In [3]:
# Test the function on the ampk_MA_double_mech model
rhs, sym_fluxes = ampk_MA_double_mech_RHS_sympyFluxVars()
S, L, invars = get_left_nullspace_vecs_invariants(rhs, sym_fluxes)
